In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 17:45:43 WARN Utils: Your hostname, TEVENO, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/07 17:45:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 17:45:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/07 17:45:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/07 17:45:47 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
spark.version

'4.1.1'

In [ ]:
Q1 HOMEWORK:

In [6]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-07 17:46:35--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.37, 3.163.140.127, 3.163.140.145, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.37|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  3.24MB/s    in 20s     

2026-03-07 17:46:55 (3.44 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [12]:
# Cargar el parquet original
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

# Guardarlo como CSV
df.write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv('fhvhv_tripdata_2025-11')

In [39]:
!ls -lh fhvhv_tripdata_2025-11.csv

ls: cannot access 'fhvhv_tripdata_2025-11.csv': No such file or directory


Q2 HOMEWORK:

In [23]:
#HOMEWORK6 
from pyspark.sql import SparkSession
import os

In [24]:
# 1. Iniciar la sesión de Spark
spark = SparkSession.builder \
    .appName("Yellow Taxi November 2025") \
    .get_builder() \
    .getOrCreate()


AttributeError: 'Builder' object has no attribute 'get_builder'

In [25]:
# 2. Leer el archivo Parquet original (Noviembre 2025)
# Asegurate de tener el archivo descargado en la ruta especificada
input_path = "yellow_tripdata_2025-11.parquet"
df = spark.read.parquet(input_path)


In [26]:
# 3. Reparticionar el DataFrame a 4 particiones
df_repartitioned = df.repartition(4)


In [27]:
# 4. Guardar a una carpeta en formato parquet
output_path = "output_yellow_2025_11"
df_repartitioned.write.parquet(output_path, mode="overwrite")


In [28]:
# 5. (Opcional) Verificar el tamaño de los archivos generados
def get_average_size(folder):
    files = [f for f in os.listdir(folder) if f.endswith('.parquet')]
    sizes = [os.path.getsize(os.path.join(folder, f)) for f in files]
    if not sizes: return 0
    return sum(sizes) / len(sizes) / (1024 * 1024) # Convertir a MB

avg_size = get_average_size(output_path)
print(f"Tamaño promedio de los archivos: {avg_size:.2f} MB")

Tamaño promedio de los archivos: 24.42 MB


In [ ]:
Q3 HOMEWORK:

In [29]:
from pyspark.sql import functions as F

In [30]:
# 1. Leer el dataset (asumiendo que ya está cargado como df)
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [31]:
# 2. Filtrar los viajes que comenzaron el 15 de noviembre
trips_15_nov = df.filter(F.to_date(df.tpep_pickup_datetime) == "2025-11-15")

In [32]:
# 3. Contar el número de registros
trip_count = trips_15_nov.count()

print(f"Total de viajes el 15 de noviembre: {trip_count}")

[Stage 7:=================================================>       (14 + 2) / 16]

Total de viajes el 15 de noviembre: 162604


In [ ]:
Q4 HOMEWORK:

In [34]:
from pyspark.sql import functions as F

In [42]:
# 1. Cargar el archivo (asegurate de que el nombre sea el correcto en tu carpeta)
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [43]:
# 2. Crear una columna con la duración en horas
# Usamos los nombres de columna sugeridos por el error: tpep_...
df_duration = df_yellow.withColumn('duration_hours', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
)

In [44]:
# 3. Encontrar el valor máximo
longest_trip = df_duration.select(F.max('duration_hours')).collect()[0][0]

print(f"El viaje más largo duró: {longest_trip:.1f} horas")

[Stage 13:==========================================>             (12 + 4) / 16]

El viaje más largo duró: 90.6 horas


In [ ]:
Q5 HOMEWORK: La interfaz de Spark (Spark UI) corre por defecto en el puerto 4040.

In [ ]:
Q6 HOMEWORK:

In [45]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-07 18:20:17--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.163.140.18, 3.163.140.127, 3.163.140.37, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.163.140.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-07 18:20:18 (180 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [46]:
from pyspark.sql import functions as F

In [47]:
# Cargar lookup de zonas
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv.1')

In [48]:
# Cargar datos de Yellow Taxi Nov 2025 (suponiendo que ya lo tenés como df_yellow)
# Si no lo tenés cargado: 
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [49]:
# Unir por el ID de la zona de recogida (PULocationID)
df_result = df_yellow.join(df_zones, df_yellow.PULocationID == df_zones.LocationID)

In [50]:
# Agrupar por zona y contar
least_frequent = df_result.groupBy('Zone') \
    .count() \
    .orderBy('count', ascending=True) \
    .limit(5)

least_frequent.show()

[Stage 19:==========================================>             (12 + 4) / 16]

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+

